# Curation Workbench\n\nNotebook-native review flow for verifier-guided candidate curation.

In [ ]:
from pathlib import Path\nimport pandas as pd\n\nfrom verifier_guided_reasoning.review import read_review_jsonl, write_review_jsonl, summarize_review_metrics

In [ ]:
INPUT_PATH = Path('data/interim/review_batch.jsonl')\nrecords = read_review_jsonl(INPUT_PATH)\nlen(records)

In [ ]:
rows = []\nfor r in records:\n    rows.append({\n        'review_id': r.review_id,\n        'prompt_id': r.prompt_id,\n        'candidate_id': r.candidate_id,\n        'prompt': r.prompt,\n        'raw_output': r.raw_output,\n        'verifier_pass': r.verifier_pass,\n        'verifier_score': r.verifier_score,\n        'curator_action': r.curator_action or '',\n        'curator_score': r.curator_score,\n        'curator_notes': r.curator_notes or '',\n    })\ndf = pd.DataFrame(rows)\ndf.head(10)

## Label Candidates\nFill `curator_action` with values like `accept`, `reject`, or `fix`, then run the next cell.

In [ ]:
# Example: mark top candidate per prompt as accepted if it has the highest verifier score.\ndf['curator_action'] = df['curator_action'].replace('', pd.NA)\nfor prompt_id, group in df.groupby('prompt_id'):\n    idx = group['verifier_score'].idxmax()\n    if pd.isna(df.loc[idx, 'curator_action']):\n        df.loc[idx, 'curator_action'] = 'accept'\ndf.head(10)

In [ ]:
updated = {row.review_id: row for row in records}\nfor row in df.itertuples(index=False):\n    record = updated[row.review_id]\n    record.curator_action = (row.curator_action if pd.notna(row.curator_action) else None)\n    record.curator_score = (float(row.curator_score) if pd.notna(row.curator_score) else None)\n    record.curator_notes = (str(row.curator_notes).strip() or None)\n\nOUTPUT_PATH = Path('data/interim/review_batch_annotated.jsonl')\nwrite_review_jsonl(updated.values(), OUTPUT_PATH)\nprint('Wrote', OUTPUT_PATH)

In [ ]:
metrics = summarize_review_metrics(updated.values())\nmetrics